 # <center> Problem Set 6 (Finetuning MACE) <center>
<center> Spring 2026 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51 <center>
<center> Due: Monday, May 11, 2026 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

### Learning Objective
The objective of this problem set is to introduce you to MACE (an equivariant neural network interatomic potential architecture) and explore its applications in molecular representation and property prediction. You will learn to extract atom-level descriptors, identify structural motifs, and compare multiple machine learning strategies (from-scratch training, foundational model finetuning, and descriptor-based feature learning). Furthermore, you will demonstrate molecular dynamics (MD) workflows powered by these advanced potentials.


### Instructions
- This problem set has two modeling tasks with several sub-questions.  This is to be completed by undergraduates only.

- To get started, make your own copy of this notebook template in Colab (e.g., "Save a copy in Drive") before editing.

    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.

- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.

- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.

    -  Numerical answers yielded from running the code should be included in an Answer Block (see next cell). 

    - We have provided print statements where numerical answers are expected.

    -  Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset1.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.


### Background
MACE is an equivariant neural network interatomic potential (NNIP) architecture, which researchers have used across a number of chemical domains to predict the energies and forces of various atomic systems [Batatia et al.](https://doi.org/10.1063/5.0297006). MACE is an equivariant architecture, meaning that when an input is rotated or reflected, the resulting directional properties, such as forces, are constrained to undergo the same transformation. Researchers have trained MACE on large swaths of different chemical spaces in an effort to offer ``foundational'' models tailored for problems in that space. The MACE model pretrained on the Materials Project data, consisting of 150k organic crystals, is referred to as MACE-MP-0; the MACE model that is trained on ~1 million conformers from organic and biomolecular systems (e.g. solvated amino acids, amino-acid ligand pairs, etc) is known as MACE-OFF [Kovács et al.](https://pubs.acs.org/doi/10.1021/jacs.4c07099). There are many versions and sizes of pretrained MACE models, which can be found in the [MACE-foundation](https://github.com/ACEsuit/mace-foundations) and [MACE-OFF](https://github.com/ACEsuit/mace-off) libraries. 

Here, we will use MACE to refer to the general strategy of using a NNIP or the model library/architecture itself, and MACE-MP-0 or MACE-OFF to refer to the pretrained models we will employ (in this pset, we will use only the `medium` sizes of both models). Throughout the PSET, you will find referring to the [MACE documentation](https://mace-docs.readthedocs.io/en/latest/) very helpful. 

Our goal for this PSET will be to first explore molecular conformers and see if we can leverage a pretrained MACE model, which have demonstrated impressive performance on simple electronic and quantum properties, to learn protein-ligand binding energy, $\Delta G_{bind}$. 

$\Delta G = E(\text{protein and ligand bound}) - E(\text{protein and ligand unbound})$ 

Though this is still a change in energy, rather than energy itself, we hypothesize that we can utilize the MACE architecture and pretraining information to learn effectively from a small subset of data. Unfortunately, protein-ligand systems are unlikely to fit in memory on a single GPU, so to simplify matters (and increase the difficulty of the learning task for MACE), we will select only one protein system--PFKFB3--and use the few ligands reported for which we have both conformer information and experimental binding affinity, which has been converted into a $\Delta G$ for our use case [Ross et al.](https://doi.org/10.1038/s42004-023-01019-9). We will refrain from supplying the protein or solvent atomic coordinates to speed up prediction as using all these relevant coordinates is too memory-intensive to load on a single GPU for training or inference. 

After exploring some of the conformer data in part 1, you will train three different models and compare performance in part 2: a MACE model from scratch, a finetuned pretrained MACE-OFF model, and a simple GNN/MLP head that takes MACE-learned descriptors as input. Part 3 will explore as an extension task running molecular dynamics with MACE as the force field instead of a more semi-empirical method.


Before starting, make sure to **request a GPU**! For this PSET, a **T4 GPU** should be sufficient to complete all problems.

## Objectives
In Problem Set 6, you'll explore working with a "foundational" machine learning interatomic potential (MLIP) model, MACE, which can be used to predict the forces acting upon or the energies of an atomic structure, with the following objectives:
* Visualizing conformers with `py3DMol` and their energies
* Querying energies and atom-level embeddings from MACE
* Comparing from-scratch vs. fine-tuning strategies for learning how to predict ΔG binding energy based on limited experimental data.
* Understanding limitations of these models

The MACE documentation highlights a lot of possible use cases, so I'd recommend reading through some of the docs to get an idea of what kinds of challenges MACE has addressed through feature availability (e.g. training different levels of theory, using cu-equivariance, MD simulations, etc); https://mace-docs.readthedocs.io/en/latest/guide/intro.html

### Download required data

In [ ]:
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-nonbio/data/pfkfb3_automap_ligands.sdf
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-nonbio/data/pfkfb3_automap_protein.pdb
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-nonbio/data/1uao.pdb
# we also need the MACE weights, but these will be autocollected when we download the calculators.

In [ ]:
# do not modify!
!uv pip install rdkit tqdm mace-torch py3Dmol umap-learn numpy==2.0.0 openbabel-wheel torch-geometric mdtraj
!uv pip install git+https://github.com/imagdau/aseMolec@main
# let's try using a version that lets us freeze certain layers
!git clone -b mace-freeze https://github.com/7radians/mace-freeze.git
!uv pip install ./mace-freeze

In [ ]:
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors,Crippen
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
import itertools
from tqdm import tqdm
import mace
import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt
import py3Dmol
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact, IntSlider
from openbabel import pybel

import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.utils import shuffle

matplotlib.rcParams.update({'font.size': 15})
matplotlib.rc('lines', linewidth=3, color='g')
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams["xtick.major.size"] = 6
matplotlib.rcParams["ytick.major.size"] = 6
matplotlib.rcParams["ytick.major.width"] = 2
matplotlib.rcParams["xtick.major.width"] = 2
matplotlib.rcParams['text.usetex'] = False

RANDOM_STATE = 421337

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Part 1: Exploring conformer distribution and utilizing MACE to look at energies (25 points)

## Part 1.1: Visualize molecule conformers & datasets <font color="blue">(7.5 points)</font>

First, let's load our dataset with `pybel`, the Python library for openbabel. For this visualization, it'll be informative to view how each of the molecules fit into the protein binding pocket, so as an auxiliary, let's also load the pdb file.


In [ ]:
mols = [m for m in pybel.readfile("sdf", "pfkfb3_automap_ligands.sdf") if m is not None]
pdb = pybel.readfile("pdb", "pfkfb3_automap_protein.pdb")

We should first visualize our molecules (or conformers thereof); here is an example with `py3Dmol`:

In [ ]:
conformer1 = """
6
Conformer 1
C 0.000 0.000 0.000
H 0.000 0.000 1.089
H 1.026 0.000 -0.363
H -0.513 0.890 -0.363
H -0.513 -0.890 -0.363
H 0.513 0.890 -0.363
"""
# set up a Javascript viewer
view = py3Dmol.view(width=800, height=400)

# add a molecule by passing the object and giving it its type.
view.addModel(conformer1, "xyz")
# objects are 0-indexed; to set the style, we can specify the model we want to change,
# then the specific visualization styles we want.
view.setStyle({'model': 0}, {"stick": {}, "sphere": {"colorscheme": "Jmol"}})

# necessary for setting viewer
view.zoomTo()
view.show()

**Task 1**: Visualize the conformers using py3Dmol, alongside the protein structure for reference (set a lower opacity for the PDB structure only).
To get the "block" representation, you can use the `.write(format)` function of a pybel molecule object.


It is encouraged to try using a Jupyter slider to make it easy to visualize all the conformers, like so:

```
slider = IntSlider(min=0, max=len(mols)-1, step=1, description="Pose:")
interact(view_mol, idx=slider)
```
And all that is needed is to define a view_mol that takes in an `idx` keyword.

In [ ]:
########## Code ###########
# your code for view_mol here

# Commented to let solutions run
# slider = IntSlider(min=0, max=len(mols)-1, step=1, description="Pose:")
# interact(view_mol, idx=slider)
########## Code ###########

**Task 2**: Describe some of the visual differences you see amongst the conformers in this dataset. What features look preserved? What atoms of the pocket appear to be in interaction with the molecules? Do you see any possible correlations between observed molecular motifs and experimental ΔG?

**Write answer here.**


## Part 1.2: Compute energies of conformers with MACE <font color="blue">(5 points)</font>

Let's analyze the energies of these conformers. The MACE library offers a handy way with their `calculators` submodule to use a pretrained energetic model, and predict their energies. Let's try!

**Task**: use the mace-mp-0 and mace_off calculations to come up with energies for each of your conformers. Plot a scatterplot between each of the molecules and report any differences, for individual molecules and about the general distribution of the data.

> **Helpful hint**: You may need to use `torch.set_default_dtype(torch.float64)` (for MACE-OFF) or `torch.set_default_dtype(torch.float32)` (for MACE-MP-0) to rectify any type errors that might arise from switching the two. This is because of a caching issue with the MACE library.

In [ ]:
########## Code ##########
from mace.calculators import mace_off, mace_mp
from ase import Atoms

def build_atom(conf):
    atoms = [a.GetSymbol() for a in conf.GetAtoms()]
    poss = [conf.GetConformer().GetAtomPosition(i) for i, _ in enumerate(conf.GetAtoms())]
    return Atoms(atoms, poss)

## Code to set up calculators and collect energies
mace_off_energies = []
mace_mp_energies = []








########## Code ##########

In [ ]:
########## Code ##########
# Code for scatterplot

########## Code ##########

**Report findings here**

## Part 1.3: Plotting atom-level descriptors/features as derived from MACE <font color="blue">(12.5 points)</font>

**Task:** As you may recall from lecture and PSET 3, GNNs build atom-specific features over successive layers; while these are usually aggregated in some manner to predict a molecule-level property, we can nonetheless extract and utilize the atom-level features as possibly informative embeddings.

MACE also offers such atom-level embeddings, known as descriptors, which you can read about more [https://mace-docs.readthedocs.io/en/latest/guide/descriptors.html](here).

**Task 1:** Following the documentation above, let's compare the utility of the MACE-OFF vs. MACE-MP-0 descriptors. Collect the physical descriptors as generated by both MACE-OFF and MACE-MP-0 separately (using `invariants_only=True`). Average embeddings over all atoms per molecule (so you have one embedding per molecule), and try clustering them with UMAP. Color the samples based on their experimental  $\Delta G$ value.


In [ ]:
########## Code ##########


########## Code ##########


**Task 2**: answer the following questions:   
1) What do you observe in terms of the clustering captured by the UMAP embeddings relative to the $\Delta G$ values? Do certain clusters capture a subset of $\Delta G$ well? Why do you think this might be the case?

2) Does one model outperform the other based on the quality of clustering or separation? What information could be useful in aiding the model to perform better on out of distribution data given atomic features?


3) Based on the quality of separation, what do you think the performance of building a regressor from these embeddings will be?

**Write answer here.**


# Part 2: Comparing different strategies for transfer learning and finetuning with MACE (40 points)

In these experiments, you’ll work through three distinct training configurations—both to familiarize yourself with varied learning strategies and to give you the opportunity to try testing different model configurations using the MACE documentation as a guide. 

Now that we've explored some of the original model's capabilities, let's try leveraging what it has learned. Our goal for this problem set will be to explore the benefits of finetuning through two different means, compared to trying to train something given no other starting information.

We'll first train a MACE model from scratch. "From scratch" means that we do _not_ use the pretrained weights and instead have to try to learn the property directly. The authors of MACE refer to MACE as the architecture; and MACE-MP-0 or MACE-OFF to describe pretrained model weights. MACE is a 3D equivariant GNN architecture, so in theory you can repurpose it to learn any property where it makes sense to utilize 3D properties. Then, we'll finetune a pretrained MACE model on this property. Lastly, we'll try using the descriptors collected in Part 1.3 to train a simple regressor on the atom features to see if that aids the modeling task.


## Part 2.1: Make train/validation/test splits for training <font color="blue">(5 points)</font>
The MACE library offers a simple CLI to handle model training with simple configurations, which we'll use so as not to modify library functions. To use the CLI functionality, we'll need to create our splits and save these to disk.

**Task**: Create the splits on the ligand dataset. We will use the `ase` library which MACE relies on to process its data, and save our ligands in an `xyz` format. To simplify some of this process, we'll first save all the conformers in an XYZ file:

In [ ]:
from rdkit import Chem

suppl = Chem.SDMolSupplier("pfkfb3_automap_ligands.sdf", removeHs=False)
shuffled_mols = shuffle([mol for mol in suppl if mol is not None], random_state=RANDOM_STATE)
with open("all_ligands.xyz", "w") as out:
    # out.write(e0s)
    for mol in shuffled_mols:
        n = mol.GetNumAtoms()
        name = mol.GetProp("_Name")
        property_line = f'''Properties=species:S:1:pos:R:3 Comp=VC(3) name={name} r_exp_dg={mol.GetProp("r_exp_dg")} pbc="F F F"'''
        xyz_block = Chem.MolToXYZBlock(mol)
        xyz_block = xyz_block.split("\n")[2:] # drop first two lines to use our property line.
        xyz_block = "\n".join(xyz_block)
        out.write(f"{n}\n{property_line}\n{xyz_block}")

Now you can use `ase` with `all_ligands.xyz`. Use the read/write functions in `ase.io` [(documentation here)](https://wiki.fysik.dtu.dk/ase/ase/io/io.html) to create a train, validation, and test split. You can assume the data is shuffled for you; use a split of 70%-10%-20% t-v-t.

In [ ]:
########## Code ##########

########## Code ##########s

## Part 2.2: Train MACE from scratch <font color="blue">(10 points)</font>

For the first experiment, let's try training a new MACE model without any pretraining.

First, let's use the basic MACE architecture to try predicting binding energy from the conformer data. The MACE library offers a handy CLI interface for training models, which we will utilize. 

**Task 1**: Set up a `config_from_scratch.yml`, which has the specifications for keyword arguments you'd like to pass to MACE. An example of one such training config, which we will need to tweak, is provided below. 

<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps6-nonbio/figures/config_sample.png" width="600px" />
</div>
<div align="center">Example of a MACE config file. </div>

To this sample config, let's make some changes. You're encouraged to look through the [MACE documentation](https://mace-docs.readthedocs.io/en/latest/examples/training_examples.html) and the [argument parsing code](https://github.com/ACEsuit/mace/blob/b5faaa076c49778fc17493edfecebcabeb960155/mace/tools/arg_parser.py) to make the right choice of settings for your model. 

    
-  Task parameters: we do not have forces or stresses to train on, so any force weights need to be set to 0/our loss should not have any force or stress terms. Consider setting the `forces_weight`, `stresses_weight`, and the `loss` accordingly. MACE also incorporates *stochastic weight averaging* with `swa`, which typically trains a different model but with a separate set of energy/force weights. You can either change the swa weights or turn it off. We will also need to provide isolated atom energies, which we can ask MACE to pre-compute empirically from our dataset with `E0s: "average"`. Because we are computing per-molecule energies, our final reporting of error can also be adjusted to not report a RMSE per atom by picking a different option for `error_table`.  
    -  MACE configuration: There are two choices of MACE that might be relevant to our use case `MACE` or `ScaleShiftMACE`, the latter of which will shift energies by mean energies and forces computed from the dataset. Since we do not have forces, though, stick with `MACE`. 
    -  Training parameters: make sure to update the `train/valid/test_file` parameters to be correct references; set the `max_num_epochs` to be 100 and `batch_size` to be 10. 

In [ ]:
%%writefile config_from_scratch.yml
# fill! remove this comment when done so that it does not get printed to the config file.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from mace.cli.run_train import main as mace_run_train_main
import sys
import logging

def train_mace(config_file_path):
    logging.getLogger().handlers.clear()
    sys.argv = ["program", "--config", config_file_path]
    mace_run_train_main()

In [ ]:
train_mace("config_from_scratch.yml")


**Task 2**: Now that we have a trained model, let's take a coarse evaluation of performance. Use the utility to produce predictions on the training, validation, and test sets (we recommend naming output files by both split and training type, to help disambiguate from later problems!), and plot a scatterplot of predicted vs. experimental values. Report the R^2 and RMSE for each split.

Use the `eval_mace` function and `aseMolec.pltProps`, `aseMolec.extAtoms` library to help you plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test  predictions from the baseline model. Label all axes and titles appropriately.

For a helpful reference using the `aseMolec` library, see here:

In [ ]:
from mace.cli.eval_configs import main as mace_eval_configs_main
import sys

def eval_mace(configs, model, output):
    sys.argv = ["program", "--configs", configs, "--model", model, "--output", output]
    mace_eval_configs_main()


########### Code ###########
# Change the Nones, and repeat for train/val/test split
# eval_mace(configs=None, # This should be your input file
#           model=None, # Path to your (stage two) model weights
#           output=None # Desired name of output file.
# )
########### Code ###########

In [ ]:
########### Code ###########
from aseMolec import pltProps as pp
from ase.io import read
import matplotlib.pyplot as plt
from aseMolec import extAtoms as ea
import numpy as np
# plotting code w/ R^2 and RMSE here
########### Code ###########

## Part 2.3: Finetune MACE-OFF on the same dataset <font color="blue">(5 points)</font>

Now, we'll try finetuning a copy of the MACE-OFF model. You can reuse much of the same config settings from your first model, though you'll need to `name` your model distinctly from the first to avoid overwriting your earlier training. 

**Task 1:** Finetune the MACE-OFF model (you can find the weights path from where it was downloaded for Part 1.2), by creating a `config_finetune.yml` and using the `train_mace` function. You will need to additionally set the `foundation_model` and `multiheads_finetuning` parameters. Since finetuning should already have sufficient learning of weights, train for only 50 epochs.  

Similar to Part 2.2, follow the steps to create a config file and train for 50 epochs. Then, create the scatterplots as you did for 2.2.

Bonus: you are welcome to try using `--freeze=n` to try freezing the first `n` layers of the model, to see if that buys you any performance. Full credit on this problem, though, would not require you to do so but you are encouraged to explore!

In [ ]:
%%writefile config_finetune.yml
# fill! remove this comment when done so that it does not get printed to the config file.

In [ ]:
train_mace("config_finetune.yml")

**Task 2**: After training, use the `eval_mace` function and `aseMolec.pltProps`, `aseMolec.extAtoms` library to help you plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test predictions from the finetuned model. Label all axes and titles appropriately.

In [ ]:
########### Code ###########
# Collect predictions
########### Code ###########

In [ ]:
########### Code ###########
# plotting code w/ R^2 and RMSE here
########### Code ###########

## Part 2.4 <font color="blue">(20 points)</font>: Train mini GNN/MLP on MACE descriptors

With the descriptors we collected in Part 1.3, we can also see if we can train a simple regressor head on these descriptors to see if they capture sufficient information.

In other words, we'll try a different strategy of learning from the pretrained model -- building a simple regressor to learn binding energy from the MACE descriptors we found in 1.3.

### 2.4.1 <font color="blue">(5 points)</font>: Preliminary questions
1. What are the conceptual differences between finetuning and using these descriptors as our input?
2. What might be the benefits of designing a predictive model separate from the MACE architecture?


**Write answer here**

## Part 2.4.2 <font color="blue">(10 points)</font>: Build a mini GNN/MLP architecture to predict $\Delta G$
Let's build a simple MLP that can operate on the atom embeddings. Using your non-averaged, **per-atom** MACE-OFF embeddings from 1.3, build a train/validation/test split, DataLoaders (of batch size 4) for your data, and a simple GNN/MLP architecture as follows:

1. (at least) one GCNConv layer between atoms that preserves the size of the embedding.
2. an MLP, composed of two linear layers with requisite nonlinearities, that operates on each embedding individually
3. a pooling step to aggregate per-atom embeddings into a single value.

Feel free to refer back to the code you wrote in PSET 3 to help you with this!

> **Helpful hints**:
> 1. Remember that for a GNN convolution, you'll need edge_indices. You can get these from the pybel molecule object with `.GetBond(idx)`.
> 2. Make sure that your descriptor and edge indices are consistent with the number of (heavy) atoms in the molecule. You may need to use `mol.DeleteHydrogens()` to help fix this.
> 3. We encourage you to use the PyTorch Geometric library as seen in PSET 3, not just for convolutions but also for the data batching with the `Data` object, which can automatically handle collations/slicing of batches for you. Take a look [here](https://pytorch-geometric.readthedocs.io/en/2.5.0/tutorial/create_dataset.html) for reference. You can return a `Data` object with the `x` and `edge_index` keywords set properly in your `Dataset.__getitem__` function.

In [ ]:
########### Code ###########
from torch_geometric.data import Data
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
torch.set_default_dtype(torch.float64) # since we are using MACE-OFF


# Set up your dataset and dataloaders, as well as your splits.

########### Code ###########

In [ ]:
########### Code ###########

class MoleculeMLP(nn.Module):
    def __init__(self):
        super().__init__()
        ### Fill

        ### Fill

    def forward(self, data):
        ### Fill
        return ### Fill

########### Code ###########

In [ ]:
########### Code ###########
# Write a training loop here.
########### Code ###########


## Part 2.4.3 <font color="blue">(5 points)</font>: Plot scatterplot of predictions

After training, plot a scatterplot, RMSE, and $R^2$ for each of the train, validation, and test predictions from the descriptors model. Label all axes and titles appropriately.


In [ ]:
########### Code ###########
# Write a prediction and scatterplot here.
########### Code ###########

## Part 2.5 <font color="blue">(5 points)</font>: Overall evaluation and observations

We just tried three different strategies for training! Report some of the differences you see in performance, any outliers or handicaps you observe in quality, and what you think overall the best strategy could be. Finally, as a conceptual question, if we had multiple protein-ligand systems that we wanted to try using the MACE architecture to train on, what would be your preferred strategy of learning binding energy?


**Write answer here.**


# Part 3  <font color="blue">(5 points)</font>: Run MD simulations with the MACE-OFF potential

As a final (fun) task, let's also explore how one can use a learned potential (e.g. MACE-OFF or MACE-MP-0) to compute molecular dynamics simulations. Typically, these are done with physics-based force fields which can be at times slow to run and therefore prohibitive to running large-scale simulations. Neural network potentials like MACE-OFF offer the opportunity to accelerate these simulations, though the sacrifice in accuracy can vary across different systems and configurations.

Let's try running a simulation! Since the full protein-ligand system is too large, we'll use a different protein (`1aou.pdb`) as an example. Load the `1aou.pdb` into the starting configuration, and then following the instructions [here](https://mace-docs.readthedocs.io/en/latest/guide/ase.html), set up a Langevin simulation to run for 500 timesteps with an interval of 25. Unnlike the tutorial, make sure to turn off periodic boundary conditions to prevent the neighbor list from exploding:
```
atoms.set_pbc(False)
```


Visualize the trajectory using a tool of your choice (e.g. PyMol locally, which you can install [here](https://ist.mit.edu/schrodinger/pymol)), and comment on any changes you see visually occurring over the course of the
trajectory.

In [ ]:
########### Code ###########
# generate your trajectory here, and save to md_protein.xyz


########### Code ###########

In [ ]:
# Demonstrate you were able to run the simulation properly:
import mdtraj as md
traj = md.load('md_protein.xyz', top='1uao.pdb')
print(len(traj)) # should be 21 frames

In [ ]:
# Code to visualize trajectory -- may be buggy
import py3Dmol
with open('md_protein.xyz', 'r') as f:
    xyz_text = f.read()


view = py3Dmol.view(width=400, height=400)
view.addModelsAsFrames(xyz_text, "xyz")
# view.addModel(xyz_text, "xyz", {'vibrate': {'frames':10,'amplitude':1}})

view.setStyle({'sphere':{'scale':0.30},'stick':{'radius':0.25}})
view.setBackgroundColor('0xeeeeee')
view.animate({'loop': 'backAndForth'})
view.zoomTo()
view.show()


**Task**: Analyze what changes you see occurring, if any, within the trajectory. If the _animation_ does not load properly, you can download the trajectory locally to visualize it with a tool like PyMol or a local jupyter notebook to visualize the trajectory. Based on your findings from earlier stages of the homework, how accurate would you expect this simulation to be? What changes do you think you could make to help improve the quality of the simulation?

**Write answer here**